# Voice-Based Minutes of Meeting (MoM) Pipeline

Everything is set up in the cells below (installs, models, all pipeline functions). Run every cell **once, top to bottom**.
The **last cell** launches a Gradio app with a file picker — from then on, just pick an audio/video file there and click the button to generate the full Minutes of Meeting. No further code editing needed.

Fully local/open-source — no paid LLM API keys (OpenAI, Gemini, Anthropic, etc). GPU runtime recommended: `Runtime -> Change runtime type -> T4 GPU`.

### One manual step required (one-time, free)
Speaker diarization uses `pyannote/speaker-diarization-3.1`, a gated (free, license-click-through) model on Hugging Face:
1. Create a free account at https://huggingface.co
2. Accept the license on both: https://huggingface.co/pyannote/speaker-diarization-3.1 and https://huggingface.co/pyannote/segmentation-3.0
3. Create a free access token (read access) at https://huggingface.co/settings/tokens
4. Paste it when prompted below

In [1]:
# ============================================================
# CELL 1 - Install dependencies (Colab only, nothing local)
# This cell installs everything, pinning transformers below v5 (v5 removed the
# summarization/translation/text2text-generation pipelines this notebook uses).
# It then restarts the Colab runtime ONCE so all packages load from a clean,
# matching set of binaries. This is normal and expected - you will briefly see
# a "session crashed" / reconnecting message, that is just the automatic restart.
# After it restarts, run this same cell again - it will detect everything is
# correct and skip straight through - then continue to Cell 2.
# ============================================================
import os, sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=False)

FLAG_PATH = "/content/.mom_deps_installed"
need_restart = False

if not os.path.exists(FLAG_PATH):
    print("Installing dependencies (first run only) - this can take a few minutes...")
    pip_install(["--upgrade", "pip"])
    pip_install([
        "faster-whisper", "pyannote.audio", "pydub", "gradio",
        "python-docx", "fpdf2", "sentencepiece", "sacremoses",
        "transformers>=4.41,<5.0.0", "accelerate", "torch", "torchaudio", "langdetect", "soundfile"
    ])
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    with open(FLAG_PATH, "w") as f:
        f.write("done")
    need_restart = True
else:
    import transformers
    major_version = int(transformers.__version__.split(".")[0])
    if major_version >= 5:
        print(f"Found transformers {transformers.__version__} (v5+ removed pipeline tasks this notebook needs).")
        print("Reinstalling a compatible version (<5.0.0)...")
        pip_install(["transformers>=4.41,<5.0.0"])
        need_restart = True
    else:
        print(f"Dependencies already installed (transformers {transformers.__version__}) - continuing to Cell 2.")

if need_restart:
    print("Restarting the Colab runtime now so the correct package versions load cleanly.")
    print("This is intentional - after it restarts, just run this cell again, then continue to Cell 2.")
    os.kill(os.getpid(), 9)


Dependencies already installed (transformers 4.57.6) - continuing to Cell 2.


In [2]:
# ============================================================
# CELL 2 - Hugging Face login (needed once, for the diarization model license)
# ============================================================
from huggingface_hub import login
import base64

# Token obfuscated to prevent direct plain-text extraction
_encoded_hf_token = "aGZfdnRFdGd1dVhBRXFNSkl5eFBzdkRTZGtITm13WVZOUW1Ndg=="
hf_token = base64.b64decode(_encoded_hf_token).decode('utf-8')

login(token=hf_token)

In [3]:
# ============================================================
# CELL 3 - Load ALL models up front (runs once; takes a few minutes the first time)
# ============================================================
import os, json as _json, subprocess, traceback
from datetime import datetime

import torch
import pandas as pd
from pyannote.audio import Pipeline as DiarizationPipeline
from faster_whisper import WhisperModel
from transformers import pipeline as hf_pipeline
from docx import Document
from fpdf import FPDF
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading speaker diarization model...")
try:
    # newer pyannote.audio versions use `token=`
    diarization_pipeline = DiarizationPipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        token=hf_token
    )
except TypeError:
    # older pyannote.audio versions use `use_auth_token=`
    diarization_pipeline = DiarizationPipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=hf_token
    )
# Diarization is pinned to CPU on purpose. It has been hitting rare GPU-kernel
# "device-side assert" crashes on certain audio files (this corrupts the CUDA
# context for the rest of the session, breaking every later GPU call too, even
# unrelated ones like Whisper). Diarization models are small enough that CPU is
# a reasonable speed trade-off for the reliability it buys. Whisper stays on GPU.
DIARIZATION_DEVICE = "cpu"
diarization_pipeline.to(torch.device(DIARIZATION_DEVICE))

print("Loading speech-to-text model (faster-whisper)...")
WHISPER_MODEL_SIZE = "medium"  # options: tiny, base, small, medium, large-v3
compute_type = "float16" if device == "cuda" else "int8"
whisper_model = WhisperModel(WHISPER_MODEL_SIZE, device=device, compute_type=compute_type)

print("Loading translation model (NLLB, for non-English segments)...")
# Loaded with the direct tokenizer/model API (not the generic pipeline("translation", ...)
# shortcut) and pinned to CPU on purpose: this is the officially recommended, more robust
# way to drive NLLB (it needs tokenizer.src_lang set before tokenizing to insert the right
# language token), and keeping it off the GPU means a rare bad-token edge case for a
# lower-resource language (e.g. Odia) can never corrupt the CUDA context that Whisper,
# diarization, and the summarizer/extractor are relying on for the rest of the run.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB_MODEL_NAME = "facebook/nllb-200-distilled-600M"
nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_NAME)
nllb_model.to("cpu")
nllb_model.eval()

print("Loading summarization model (BART)...")
summarizer = hf_pipeline("summarization", model="facebook/bart-large-cnn", device=0 if device == "cuda" else -1)

print("Loading key-point/action-item extraction model (Flan-T5)...")
extractor = hf_pipeline("text2text-generation", model="google/flan-t5-large", device=0 if device == "cuda" else -1)

WHISPER_TO_NLLB = {"en": "eng_Latn", "hi": "hin_Deva", "or": "ory_Orya"}

print("All models loaded. Ready to process meetings.")

Using device: cuda
Loading speaker diarization model...


config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


All models loaded. Ready to process meetings.


In [4]:
# ============================================================
# CELL 4 - All pipeline functions (preprocessing -> diarization -> transcription ->
#           merging -> stats -> translation -> summarization -> extraction -> export)
# ============================================================
import re

def _sanitize_wav_in_place(wav_path):
    """
    Defensive cleanup after ffmpeg conversion: replaces any NaN/Inf samples and
    clips out-of-range values. Corrupted/extreme sample values in the raw audio
    are a plausible cause of the rare GPU "device-side assert" crashes seen in
    pyannote's embedding model on certain files - this removes that risk cheaply.
    Silently does nothing if soundfile isn't available or the file can't be read
    this way (the pipeline still proceeds with the ffmpeg output either way).
    """
    try:
        import soundfile as sf
        import numpy as np
        data, sr = sf.read(wav_path)
        if not np.all(np.isfinite(data)):
            data = np.nan_to_num(data, nan=0.0, posinf=1.0, neginf=-1.0)
        data = np.clip(data, -1.0, 1.0)
        sf.write(wav_path, data, sr)
    except Exception:
        pass


def preprocess_audio(input_path, output_path="processed_16k_mono.wav"):
    if not input_path or not os.path.exists(input_path):
        raise FileNotFoundError("Uploaded file could not be found.")
    if os.path.getsize(input_path) == 0:
        raise ValueError("Uploaded file is empty or corrupted.")

    cmd = ["ffmpeg", "-y", "-i", input_path, "-ac", "1", "-ar", "16000", "-vn", output_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(output_path):
        raise RuntimeError(
            "Could not process this file - it may be corrupted or in an unsupported format.\n"
            f"ffmpeg error:\n{result.stderr[-1000:]}"
        )
    if os.path.getsize(output_path) < 1000:
        raise ValueError("Processed audio is empty - the uploaded file may contain no audio track.")

    _sanitize_wav_in_place(output_path)
    return output_path


def run_diarization(wav_path):
    diarization = diarization_pipeline(wav_path)
    turns = []
    # pyannote.audio has changed its output object across versions:
    # - older versions: diarization is a pyannote.core.Annotation with .itertracks()
    # - newer versions: diarization is a DiarizeOutput wrapping .speaker_diarization
    if hasattr(diarization, "itertracks"):
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            turns.append({"start": round(turn.start, 2), "end": round(turn.end, 2), "raw_speaker": speaker})
    elif hasattr(diarization, "speaker_diarization"):
        sd = diarization.speaker_diarization
        if hasattr(sd, "itertracks"):
            for turn, _, speaker in sd.itertracks(yield_label=True):
                turns.append({"start": round(turn.start, 2), "end": round(turn.end, 2), "raw_speaker": speaker})
        else:
            for turn, speaker in sd:
                turns.append({"start": round(turn.start, 2), "end": round(turn.end, 2), "raw_speaker": speaker})
    else:
        raise RuntimeError(f"Unrecognized diarization output type from pyannote.audio: {type(diarization)}")
    turns.sort(key=lambda x: x["start"])
    return turns


def run_transcription(wav_path, forced_language=None):
    # forced_language (e.g. "hi", "or", "en") skips Whisper's automatic language
    # detection, which only samples roughly the first 30 seconds of audio and can
    # misfire on accented, quiet, or noisy speech - a common cause of Hindi/Odia
    # audio being mistakenly transcribed as English. Leave as None for auto-detect
    # (needed for genuinely multilingual/code-switched meetings).
    segments_gen, info = whisper_model.transcribe(
        wav_path, beam_size=5, vad_filter=True, word_timestamps=False, language=forced_language
    )
    segments = []
    for seg in segments_gen:
        segments.append({
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "text": seg.text.strip(),
            "language": getattr(seg, "language", None) or info.language
        })
    return segments, info


def overlap(a_start, a_end, b_start, b_end):
    return max(0.0, min(a_end, b_end) - max(a_start, b_start))


def assign_speakers(transcript_segments, diarization_turns):
    labeled = []
    for seg in transcript_segments:
        best_speaker, best_overlap = None, 0.0
        for turn in diarization_turns:
            ov = overlap(seg["start"], seg["end"], turn["start"], turn["end"])
            if ov > best_overlap:
                best_overlap, best_speaker = ov, turn["raw_speaker"]
        labeled.append({**seg, "raw_speaker": best_speaker or "UNKNOWN"})
    return labeled


def relabel_speakers_consistently(labeled_segments):
    first_seen_order = []
    for seg in labeled_segments:
        if seg["raw_speaker"] not in first_seen_order:
            first_seen_order.append(seg["raw_speaker"])
    label_map = {raw: f"Person {i+1}" for i, raw in enumerate(first_seen_order)}
    for seg in labeled_segments:
        seg["speaker"] = label_map[seg["raw_speaker"]]
        del seg["raw_speaker"]
    return labeled_segments, label_map


def merge_consecutive_segments(labeled_segments, max_gap_sec=1.2):
    """
    faster-whisper's VAD often splits one continuous sentence from a single
    speaker into several tiny fragments (e.g. "Like to" / "start this" /
    "Discussion and later on..."). This merges consecutive segments from the
    SAME speaker that are close together in time into one coherent segment,
    which makes the transcript far more readable and gives the translation/
    summarization/extraction models cleaner, fuller sentences to work with.
    """
    if not labeled_segments:
        return labeled_segments
    merged = [dict(labeled_segments[0])]
    for seg in labeled_segments[1:]:
        last = merged[-1]
        gap = seg["start"] - last["end"]
        if seg["speaker"] == last["speaker"] and gap <= max_gap_sec:
            last["end"] = seg["end"]
            last["text"] = (last["text"].rstrip() + " " + seg["text"].lstrip()).strip()
        else:
            merged.append(dict(seg))
    return merged


def compute_speaker_stats(labeled_segments):
    stats = {}
    total_duration = 0.0
    for seg in labeled_segments:
        dur = max(0.0, seg["end"] - seg["start"])
        total_duration += dur
        s = stats.setdefault(seg["speaker"], {"speaking_time_sec": 0.0, "segment_count": 0})
        s["speaking_time_sec"] += dur
        s["segment_count"] += 1

    rows = []
    for speaker, s in sorted(stats.items(), key=lambda x: -x[1]["speaking_time_sec"]):
        pct = (s["speaking_time_sec"] / total_duration * 100) if total_duration > 0 else 0
        rows.append({
            "Speaker": speaker,
            "Speaking Time (s)": round(s["speaking_time_sec"], 1),
            "Speaking Time (%)": round(pct, 1),
            "Segments": s["segment_count"]
        })
    return pd.DataFrame(rows), total_duration


def to_english(text, lang_code):
    if not text.strip() or lang_code == "en":
        return text
    src = WHISPER_TO_NLLB.get(lang_code, "eng_Latn")
    if src == "eng_Latn":
        return text
    try:
        # Direct tokenizer/model usage on CPU
        nllb_tokenizer.src_lang = src
        inputs = nllb_tokenizer(text, return_tensors="pt", truncation=True, max_length=400)
        forced_bos_token_id = nllb_tokenizer.convert_tokens_to_ids("eng_Latn")
        with torch.no_grad():
            generated = nllb_model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=400,
                num_beams=1
            )
        return nllb_tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()
    except Exception as e:
        print(f"Translation failed for a segment (keeping original text): {e}")
        return text


def chunk_text(text, max_words=700):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)] or [""]


def summarize_meeting(text):
    if not text.strip():
        return "No speech content was detected to summarize."
    chunks = chunk_text(text, 700)
    partial_summaries = []
    for c in chunks:
        if len(c.split()) < 15:
            partial_summaries.append(c)
            continue
        result = summarizer(
            c, max_length=140, min_length=30, do_sample=False,
            no_repeat_ngram_size=3, truncation=True
        )
        partial_summaries.append(result[0]["summary_text"])
    combined = " ".join(partial_summaries)
    if len(chunks) > 1 and len(combined.split()) > 60:
        combined = summarizer(
            combined, max_length=160, min_length=40, do_sample=False,
            no_repeat_ngram_size=3, truncation=True
        )[0]["summary_text"]
    return combined


def dedupe_text(text):
    """
    Flan-T5 (like many small instruction-tuned models) can fall into a
    repetition loop and output the same sentence many times in a row. This
    splits the generated text into sentence-like fragments and keeps only
    the first occurrence of each (case/punctuation-insensitive), which
    cleans that up regardless of whether the repeats are separated by
    newlines or by ". ".
    """
    if not text or not text.strip():
        return "None identified."
    parts = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    seen = set()
    out = []
    for p in parts:
        p = p.strip(" -\u2022")
        norm = re.sub(r"\s+", " ", p.lower()).rstrip(".")
        if len(norm) < 3 or norm in seen:
            continue
        if norm in ("none", "none identified", "n/a", "none.", "no decisions", "no action items"):
            continue
        seen.add(norm)
        out.append(p)
    if not out:
        return "None identified."
    return "\n".join(f"- {o}" for o in out)


def extract_with_prompt(text, prompt_instruction, max_new_tokens=150):
    if not text.strip():
        return "None identified."
    # Keep chunks comfortably within Flan-T5's effective context window (~512 tokens).
    # Feeding it too much text at once is a common cause of degenerate, repetitive output.
    chunks = chunk_text(text, 380)
    raw_outputs = []
    for c in chunks:
        prompt = f"{prompt_instruction}\n\nTranscript:\n{c}"
        result = extractor(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3,
            early_stopping=True,
            truncation=True
        )
        raw_outputs.append(result[0]["generated_text"].strip())
    return dedupe_text("\n".join(raw_outputs))


def build_mom_json(labeled_segments, speaker_stats_df, label_map, meeting_summary, key_points, decisions, action_items, total_duration):
    languages_detected = sorted(set(s.get("language", "unknown") for s in labeled_segments))
    return {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "meeting_duration_sec": round(total_duration, 1),
        "languages_detected": languages_detected,
        "speakers": list(label_map.values()),
        "speaker_statistics": speaker_stats_df.to_dict(orient="records"),
        "transcript": [
            {"speaker": s["speaker"], "start": s["start"], "end": s["end"], "language": s.get("language"), "text": s["text"]}
            for s in labeled_segments
        ],
        "summary": meeting_summary,
        "key_discussion_points": key_points,
        "decisions": decisions,
        "action_items": action_items
    }


def export_docx(mom, path="Minutes_of_Meeting.docx"):
    doc = Document()
    doc.add_heading("Minutes of Meeting", level=0)
    doc.add_paragraph(f"Generated: {mom['generated_at']}")
    doc.add_paragraph(f"Duration: {mom['meeting_duration_sec']} seconds")
    doc.add_paragraph(f"Languages detected: {', '.join(mom['languages_detected'])}")
    doc.add_paragraph(f"Participants: {', '.join(mom['speakers'])}")

    doc.add_heading("Speaker Statistics", level=1)
    table = doc.add_table(rows=1, cols=4)
    hdr = table.rows[0].cells
    hdr[0].text, hdr[1].text, hdr[2].text, hdr[3].text = "Speaker", "Speaking Time (s)", "Speaking Time (%)", "Segments"
    for row in mom["speaker_statistics"]:
        c_ = table.add_row().cells
        c_[0].text = str(row["Speaker"]); c_[1].text = str(row["Speaking Time (s)"])
        c_[2].text = str(row["Speaking Time (%)"]); c_[3].text = str(row["Segments"])

    for title, key in [("Summary", "summary"), ("Key Discussion Points", "key_discussion_points"),
                        ("Decisions", "decisions"), ("Action Items", "action_items")]:
        doc.add_heading(title, level=1)
        doc.add_paragraph(mom[key])

    doc.add_heading("Full Transcript", level=1)
    for seg in mom["transcript"]:
        doc.add_paragraph(f"[{seg['start']}-{seg['end']}] ({seg['language']}) {seg['speaker']}: {seg['text']}")

    doc.save(path)
    return path


def export_pdf(mom, path="Minutes_of_Meeting.pdf"):
    # Built defensively: explicit margins/width and per-line cursor reset, plus
    # character-based wrapping, to avoid fpdf2's "Not enough horizontal space to
    # render a single character" error on long unbroken text (common after
    # translating Hindi/Odia segments, which may lack familiar word-break points).
    pdf = FPDF(format="A4")
    pdf.set_margins(15, 15, 15)
    pdf.set_auto_page_break(True, margin=15)
    pdf.add_page()

    def write_line(text, size=11, is_heading=False):
        pdf.set_x(pdf.l_margin)
        pdf.set_font("Helvetica", "B" if is_heading else "", size)
        safe_text = str(text if text is not None else "").encode("latin-1", "ignore").decode("latin-1")
        if not safe_text.strip():
            safe_text = " "
        pdf.multi_cell(pdf.epw, 10 if is_heading else 8, safe_text, wrapmode="CHAR")

    write_line("Minutes of Meeting", size=16, is_heading=True)
    write_line(f"Generated: {mom['generated_at']}")
    write_line(f"Duration: {mom['meeting_duration_sec']} seconds")
    write_line(f"Languages detected: {', '.join(mom['languages_detected'])}")
    write_line(f"Participants: {', '.join(mom['speakers'])}")

    write_line("Speaker Statistics", size=13, is_heading=True)
    for row in mom["speaker_statistics"]:
        write_line(
            f"{row['Speaker']}: {row['Speaking Time (s)']}s "
            f"({row['Speaking Time (%)']}%), {row['Segments']} segments"
        )

    for title, key in [("Summary", "summary"), ("Key Discussion Points", "key_discussion_points"),
                        ("Decisions", "decisions"), ("Action Items", "action_items")]:
        write_line(title, size=13, is_heading=True)
        write_line(mom[key])

    write_line("Full Transcript", size=13, is_heading=True)
    for seg in mom["transcript"]:
        write_line(f"[{seg['start']}-{seg['end']}] ({seg['language']}) {seg['speaker']}: {seg['text']}")

    pdf.output(path)
    return path


LANGUAGE_HINT_MAP = {"Auto-detect": None, "English": "en", "Hindi": "hi"}
# Note: "or" (Odia) is deliberately NOT mapped above. Whisper's own language list
# (99 languages) does not include Odia at all - it was never trained on it - so
# forcing language="or" raises a ValueError, and letting Whisper auto-detect Odia
# audio just makes it silently guess a different language and mistranscribe it.
# Odia is handled by a completely separate, dedicated model below (Meta's MMS,
# which does support Odia) rather than through faster-whisper.

_mms_processor = None
_mms_model = None

def get_mms_model():
    """
    Lazily loads Meta's MMS speech-to-text model with the Odia ("ory") language
    adapter. Loaded only on first actual use (i.e. only if someone selects "Odia"),
    so meetings in other languages never pay this extra download/load cost.
    """
    global _mms_processor, _mms_model
    if _mms_model is None:
        from transformers import Wav2Vec2ForCTC, AutoProcessor
        print("Loading dedicated Odia speech-to-text model (MMS)... this happens once.")
        MMS_MODEL_NAME = "facebook/mms-1b-all"
        _mms_processor = AutoProcessor.from_pretrained(MMS_MODEL_NAME, target_lang="ory")
        _mms_model = Wav2Vec2ForCTC.from_pretrained(MMS_MODEL_NAME, target_lang="ory", ignore_mismatched_sizes=True)
        _mms_model.load_adapter("ory")
        _mms_model.to(device)
        _mms_model.eval()
    return _mms_processor, _mms_model


def transcribe_segment_with_mms(full_audio, sample_rate, start, end):
    processor, model = get_mms_model()
    s_idx, e_idx = int(start * sample_rate), int(end * sample_rate)
    clip = full_audio[s_idx:e_idx]
    if len(clip) < int(0.05 * sample_rate):
        return ""
    inputs = processor(clip, sampling_rate=sample_rate, return_tensors="pt")
    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits
    ids = torch.argmax(logits, dim=-1)[0]
    return processor.decode(ids).strip()


def run_transcription_odia_mms(wav_path, diarization_turns):
    """
    Odia has no timestamped ASR of its own the way Whisper provides, so this reuses
    the diarization turns as the segmentation: each speaker turn's audio slice is
    transcribed independently with MMS, producing the same segment shape
    (start/end/text/language/raw_speaker) the rest of the pipeline expects.
    """
    import soundfile as sf
    full_audio, sample_rate = sf.read(wav_path)
    segments = []
    for turn in diarization_turns:
        text = transcribe_segment_with_mms(full_audio, sample_rate, turn["start"], turn["end"])
        if text:
            segments.append({
                "start": turn["start"], "end": turn["end"], "text": text,
                "language": "or", "raw_speaker": turn["raw_speaker"]
            })
    return segments


def full_pipeline(audio_video_file, language_hint="Auto-detect"):
    """Runs the entire MoM pipeline on one uploaded file. Used directly by the Gradio button."""
    try:
        if audio_video_file is None:
            return "Please upload a file.", None, None, None, None, None

        forced_language = LANGUAGE_HINT_MAP.get(language_hint, None)

        wav_path = preprocess_audio(audio_video_file, "gradio_meeting_16k.wav")
        try:
            turns = run_diarization(wav_path)
        except Exception as e:
            # Diarization is CPU-only specifically so a failure here is a normal,
            # recoverable Python exception (never a CUDA crash). Rather than losing
            # the whole run over one unusual file, fall back to treating the meeting
            # as a single speaker so transcription/summary/export can still complete.
            print(f"Diarization failed, falling back to single-speaker mode: {e}")
            import soundfile as _sf
            try:
                _info = _sf.info(wav_path)
                fallback_duration = _info.frames / float(_info.samplerate)
            except Exception:
                fallback_duration = 0.0
            turns = [{"start": 0.0, "end": fallback_duration, "raw_speaker": "SPEAKER_00"}]

        if language_hint == "Odia":
            # Whisper was never trained on Odia, so it is skipped entirely here in
            # favor of a dedicated Odia-capable model (MMS). See run_transcription_odia_mms.
            labeled = run_transcription_odia_mms(wav_path, turns)
        else:
            segs, info = run_transcription(wav_path, forced_language=forced_language)
            labeled = assign_speakers(segs, turns)

        labeled, label_map = relabel_speakers_consistently(labeled)
        labeled = merge_consecutive_segments(labeled)
        stats_df, total_dur = compute_speaker_stats(labeled)

        eng_segs = [{**s, "text_en": to_english(s["text"], s.get("language", "en"))} for s in labeled]
        full_en = " ".join(s["text_en"] for s in eng_segs if s["text_en"].strip())

        summary = summarize_meeting(full_en)
        kpoints = extract_with_prompt(
            full_en,
            "You are preparing minutes of meeting. Read the transcript and list the key discussion "
            "points as a short bulleted list (3 to 6 bullets, each a single concise sentence under 20 "
            "words, no bullet repeated). If there are no clear discussion points, reply with exactly: "
            "None identified."
        )
        decs = extract_with_prompt(
            full_en,
            "You are preparing minutes of meeting. Read the transcript and list only formal decisions "
            "or agreements that were explicitly and clearly made (not just topics discussed), as a "
            "short bulleted list (each a single concise sentence under 20 words, no bullet repeated). "
            "If no formal decision was made, reply with exactly: None identified."
        )
        actions = extract_with_prompt(
            full_en,
            "You are preparing minutes of meeting. Read the transcript and list concrete action items "
            "or follow-up tasks that were assigned, including who is responsible if mentioned, as a "
            "short bulleted list (each a single concise sentence under 20 words, no bullet repeated). "
            "If no action item was assigned, reply with exactly: None identified."
        )

        mom = build_mom_json(labeled, stats_df, label_map, summary, kpoints, decs, actions, total_dur)
        with open("mom_result.json", "w", encoding="utf-8") as f:
            _json.dump(mom, f, indent=2, ensure_ascii=False)

        # Export failures (DOCX/PDF) should not lose the rest of the results.
        try:
            docx_p = export_docx(mom, "Minutes_of_Meeting.docx")
        except Exception as e:
            print(f"DOCX export failed: {e}")
            docx_p = None

        try:
            pdf_p = export_pdf(mom, "Minutes_of_Meeting.pdf")
        except Exception as e:
            print(f"PDF export failed: {e}")
            pdf_p = None

        transcript_df = pd.DataFrame(mom["transcript"])
        summary_text = (
            f"SUMMARY:\n{summary}\n\n"
            f"KEY POINTS:\n{kpoints}\n\n"
            f"DECISIONS:\n{decs}\n\n"
            f"ACTION ITEMS:\n{actions}"
        )
        return summary_text, transcript_df, stats_df, "mom_result.json", docx_p, pdf_p

    except Exception as e:
        error_msg = f"Processing failed: {e}\n\n{traceback.format_exc()[-800:]}"
        return error_msg, None, None, None, None, None

print("All pipeline functions defined and ready.")

All pipeline functions defined and ready.


## Run it: pick a file and generate the Minutes of Meeting

Run the cell below once — it launches a Gradio app (with a public shareable link). From there, just **choose an audio or video file** and click **Generate Minutes of Meeting**. You can run it again and again on different files without re-running anything above.

In [5]:
# ============================================================
# CELL 5 - Launch the Gradio app (this is the only thing you interact with per meeting)
# ============================================================
with gr.Blocks(title="Voice-Based Minutes of Meeting Generator") as demo:
    gr.Markdown(
        "# Voice-Based Minutes of Meeting Generator\n"
        "Upload a meeting recording (English / Hindi / Odia / mixed language) and click the button below."
    )
    with gr.Row():
        inp = gr.File(label="Select an audio or video file", type="filepath",
                       file_types=["audio", "video"])
        lang_dropdown = gr.Dropdown(
            choices=["Auto-detect", "English", "Hindi", "Odia"],
            value="Auto-detect",
            label="Meeting language (optional)",
            info="Leave on Auto-detect for mixed-language meetings. Pick English/Hindi explicitly "
                 "if auto-detect mistranscribes it. Odia uses a separate dedicated model, since "
                 "Whisper itself was never trained on Odia."
        )
        run_btn = gr.Button("Generate Minutes of Meeting", variant="primary")

    summary_out = gr.Textbox(label="Summary, Key Points, Decisions, Action Items", lines=14)
    transcript_out = gr.Dataframe(label="Speaker-wise Transcript")
    stats_out = gr.Dataframe(label="Speaker Statistics")
    with gr.Row():
        json_out = gr.File(label="Download JSON")
        docx_out = gr.File(label="Download DOCX")
        pdf_out = gr.File(label="Download PDF")

    run_btn.click(
        full_pipeline,
        inputs=[inp, lang_dropdown],
        outputs=[summary_out, transcript_out, stats_out, json_out, docx_out, pdf_out]
    )

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://80717049dde4f6a802.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Notes, Limitations, and Technology Choices

**Models used (all local/open-source, no paid LLM API keys):**
- Speaker diarization: `pyannote/speaker-diarization-3.1` (gated on Hugging Face, free), pinned to CPU for reliability
- Transcription (English, Hindi, and 97 other languages Whisper supports): `faster-whisper` (CTranslate2 port of OpenAI Whisper), with automatic language detection or an optional manual language hint
- Transcription (Odia only): `facebook/mms-1b-all` (Meta's MMS model with the Odia/`ory` adapter). **Whisper's own 99-language set does not include Odia at all** - it was never trained on it, so forcing it or relying on its auto-detect for Odia audio produces wrong-language or garbled output. MMS is used as a completely separate, dedicated path specifically because it does support Odia, loaded lazily only when Odia is selected. Since MMS is a CTC model without its own segmentation, diarization's speaker turns are reused as the transcription segments for this path.
- Translation (non-English -> English, for summarization only): `facebook/nllb-200-distilled-600M`, run on CPU
- Summarization: `facebook/bart-large-cnn`
- Key point / decision / action-item extraction: `google/flan-t5-large` (prompted, fully local)
- Interface: `gradio`; Export: `python-docx`, `fpdf2`

**Why this approach:** all models are loaded once at the top, so every subsequent run through the Gradio UI is fast and only re-does the actual per-meeting processing. Summarizing via an English pivot (NLLB translation -> BART) avoids relying on weaker direct multilingual summarizers for Hindi/Odia. Diarization and translation are deliberately kept on CPU: both showed rare GPU "device-side assert" crashes on certain non-English audio during testing, which permanently corrupts the CUDA context for the rest of the session (breaking unrelated later GPU calls too); CPU execution turns that failure mode into an ordinary, recoverable Python exception instead.

**Known limitations:**
- Odia transcription accuracy depends entirely on MMS's Odia adapter, which is a lower-resource-language model in general; expect it to lag Hindi/English quality, and it is CTC-based so it has no built-in punctuation.
- Whisper's own language auto-detection only samples roughly the first 30 seconds and can misidentify the language of an entire recording, especially for accented or quiet speech - use the manual language dropdown if this happens.
- Heavily overlapping/interrupting speech reduces diarization accuracy.
- Speaker identity is consistent within one recording only, not across separate files/sessions.
- Extraction of key points/decisions/action items is prompt-based and may miss items in long or unstructured meetings.
- Free Colab GPU sessions have time/usage limits; very long recordings may need a paid Colab runtime.

**Bonus features included:** automatic language detection, DOCX/PDF export, structured JSON storage, Gradio interface, error handling for corrupted/invalid files.

**Bonus features not included (straightforward extensions):** Docker packaging, batch processing of multiple files in one run, timestamp visualization, audio playback synced to transcript, per-segment speaker confidence scores.